# 💍 Jewellery Try-On Generator: Qwen-Image-Edit-2511 on Kaggle

Takes **(model photo + jewellery product photo + prompt)** and generates a photo of the model wearing that jewellery.
It does the same thing as the Hugging Face Space, but it runs in batches, so you can make hundreds of images without the Space's time limit.

### Kaggle settings (do these once)
1. **Settings → Accelerator → `GPU T4 x2`** (or a bigger GPU if your account offers one).
2. **Settings → Internet → On** (needed to download the model).
3. **Add-ons → Secrets → Add secret** named `HF_TOKEN` with a *read* token from https://huggingface.co/settings/tokens.
   Don't paste the token into a cell.
4. **Add Input → Upload dataset** with this layout (the names are up to you):
```
jewellery-tryon-inputs/
├── models/      woman_01.jpg, woman_02.webp, ...   (photos WITHOUT jewellery)
├── jewellery/   necklace_set_01.jpg, ring_07.png, ...
└── jobs.csv     (optional, see section 5)
```

### Why the Space code doesn't run directly on Kaggle
The Space loads the **full BF16 model**: a 41 GB transformer plus a 16.6 GB text encoder, about 58 GB in total, on an 80 GB+ ZeroGPU.
A Kaggle T4 has **16 GB**. This notebook loads a **4-bit** copy of the same model (about 17 GB in total) and moves each part onto the GPU only when it's needed.

## 1. Install

In [ ]:
# Qwen-Image-Edit-2511 needs a recent diffusers (the Space pins a git commit), so install from GitHub.
!pip install -q -U "git+https://github.com/huggingface/diffusers" transformers accelerate peft bitsandbytes gguf safetensors sentencepiece

## 2. Config: edit this cell

In [ ]:
import os, glob

# ---- where your images are ----
INPUT_ROOT = "/kaggle/input"          # the notebook searches below this for models/ and jewellery/ folders
OUTPUT_DIR = "/kaggle/working/outputs"

# ---- model backend ----
#  "nf4"  : ovedrive/Qwen-Image-Edit-2511-4bit. Easiest, ~17 GB download. Licence is CC-BY-NC-SA (NON-commercial).
#  "gguf" : unsloth GGUF Q4_K_M transformer + original text encoder quantised to 4-bit while loading.
#           Everything is Apache-2.0, so commercial use is OK. ~30 GB download.
BACKEND = "nf4"

# ---- speed ----
USE_LIGHTNING = True     # 8-step distilled LoRA: about 10x faster. Turn off if quality looks worse.
LIGHTNING_STEPS = 8      # 8 or 4
STEPS_NO_LIGHTNING = 30  # used only when USE_LIGHTNING=False (very slow on T4)
CFG_NO_LIGHTNING = 4.0   # true_cfg_scale without Lightning (the Space uses 4.0)

TARGET_PIXELS = 1024 * 1024   # output area. Lower it (e.g. 768*768) if you hit OOM or it's too slow
IMAGES_PER_JOB = 1            # generate several and pick the best (each costs time)
BASE_SEED = 42

# T4/P100 have no fast bf16 attention kernel, so 12k-token attention falls back to a slow,
# memory-hungry path. This option runs only the attention op in fp16, which keeps memory low.
FP16_ATTENTION_ON_OLD_GPUS = True

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. HF login + GPU check

In [ ]:
import torch
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=os.environ["HF_TOKEN"])
    print("✅ logged in to Hugging Face")
except Exception as e:
    print("⚠️  No HF_TOKEN secret found. Public downloads still work but are rate-limited.", e)

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU{i}: {p.name}  {p.total_memory/1e9:.1f} GB  sm_{p.major}{p.minor}")
!free -g | head -2
!df -h /root /kaggle/working | tail -2

In [ ]:
# fp16-attention patch for pre-Ampere GPUs (T4 = sm_75, P100 = sm_60)
import torch.nn.functional as F
if FP16_ATTENTION_ON_OLD_GPUS and torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 8 \
        and not getattr(F, "_jewel_patched", False):
    _orig_sdpa = F.scaled_dot_product_attention
    def _h(t):
        return t.half() if torch.is_tensor(t) and t.dtype == torch.bfloat16 else t
    def _sdpa_fp16(query, key, value, *args, **kwargs):
        if query.dtype == torch.bfloat16 and query.is_cuda:
            out = _orig_sdpa(_h(query), _h(key), _h(value), *[_h(a) for a in args],
                             **{n: _h(v) for n, v in kwargs.items()})
            return out.to(torch.bfloat16)
        return _orig_sdpa(query, key, value, *args, **kwargs)
    F.scaled_dot_product_attention = _sdpa_fp16
    torch.nn.functional.scaled_dot_product_attention = _sdpa_fp16
    F._jewel_patched = True
    print("✅ fp16 attention patch active")

## 4. Load the pipeline (first run downloads ~17–30 GB, which takes 5–15 min)

In [ ]:
import math, gc
from diffusers import QwenImageEditPlusPipeline, FlowMatchEulerDiscreteScheduler

dtype = torch.bfloat16

if BACKEND == "nf4":
    pipe = QwenImageEditPlusPipeline.from_pretrained("ovedrive/Qwen-Image-Edit-2511-4bit", torch_dtype=dtype)

elif BACKEND == "gguf":
    from diffusers import QwenImageTransformer2DModel, GGUFQuantizationConfig
    from transformers import Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
    transformer = QwenImageTransformer2DModel.from_single_file(
        "https://huggingface.co/unsloth/Qwen-Image-Edit-2511-GGUF/blob/main/qwen-image-edit-2511-Q4_K_M.gguf",
        quantization_config=GGUFQuantizationConfig(compute_dtype=dtype),
        config="Qwen/Qwen-Image-Edit-2511", subfolder="transformer", torch_dtype=dtype,
    )
    text_encoder = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        "Qwen/Qwen-Image-Edit-2511", subfolder="text_encoder", torch_dtype=dtype,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                               bnb_4bit_compute_dtype=dtype),
    )
    pipe = QwenImageEditPlusPipeline.from_pretrained(
        "Qwen/Qwen-Image-Edit-2511", transformer=transformer, text_encoder=text_encoder, torch_dtype=dtype)
else:
    raise ValueError(BACKEND)

if USE_LIGHTNING:
    # Scheduler settings recommended by the Qwen-Image-Lightning authors
    pipe.scheduler = FlowMatchEulerDiscreteScheduler.from_config({
        "base_image_seq_len": 256, "base_shift": math.log(3), "invert_sigmas": False,
        "max_image_seq_len": 8192, "max_shift": math.log(3), "num_train_timesteps": 1000,
        "shift": 1.0, "shift_terminal": None, "stochastic_sampling": False,
        "time_shift_type": "exponential", "use_beta_sigmas": False, "use_dynamic_shifting": True,
        "use_exponential_sigmas": False, "use_karras_sigmas": False,
    })
    try:
        pipe.load_lora_weights(
            "lightx2v/Qwen-Image-Edit-2511-Lightning",
            weight_name=f"Qwen-Image-Edit-2511-Lightning-{LIGHTNING_STEPS}steps-V1.0-bf16.safetensors")
        print(f"✅ Lightning {LIGHTNING_STEPS}-step LoRA loaded")
    except Exception as e:
        print("⚠️  Could not load Lightning LoRA on this quantised model, so running the normal schedule instead:", e)
        USE_LIGHTNING = False
        pipe.scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained("Qwen/Qwen-Image-Edit-2511", subfolder="scheduler")

# 16 GB isn't enough to hold text encoder + transformer + VAE at the same time, so move each
# one to the GPU only when it's used.
pipe.enable_model_cpu_offload()
try:
    pipe.vae.enable_tiling()
except Exception:
    pass
pipe.set_progress_bar_config(disable=False)

STEPS = LIGHTNING_STEPS if USE_LIGHTNING else STEPS_NO_LIGHTNING
CFG   = 1.0 if USE_LIGHTNING else CFG_NO_LIGHTNING   # Lightning is distilled: no CFG, which also halves compute
gc.collect(); torch.cuda.empty_cache()
print("steps:", STEPS, "true_cfg_scale:", CFG)

## 5. Prompts

**Image order matters.** This notebook always passes **Picture 1 = model photo** and **Picture 2 = jewellery photo**, and the prompt refers to them that way.

Tips from comparing your two test images:
* Name **every piece** (necklace + pendant + earrings) and **where** each one goes.
* Ask for **"realistic, delicate, true-to-scale"**. Otherwise the model tends to make pendants oversized and gold look like yellow plastic.
* Describe the **metal** (22k yellow gold, two-tone yellow/white gold, rose gold, kundan, diamonds…).
* Say what must **not** change (face, hair, outfit, background).

In [ ]:
PROMPT_TEMPLATES = {
"necklace_set": (
    "Make the woman in Picture 1 wear the exact jewellery set shown in Picture 2: put the necklace with its pendant "
    "around her neck, resting naturally on her collarbone, and put the matching drop earrings on both earlobes. "
    "Reproduce the jewellery design from Picture 2 exactly: same shapes, same engraved texture, same {metal} finish, "
    "same chain, at realistic delicate true-to-life scale with natural metallic reflections and soft shadows on the skin. "
    "Keep her face, identity, expression, hairstyle, outfit, pose, lighting and background completely unchanged. "
    "Photorealistic luxury jewellery catalogue photograph, sharp focus on the jewellery."),
"necklace": (
    "Make the woman in Picture 1 wear the exact necklace from Picture 2 around her neck, resting naturally on her collarbone. "
    "Keep the design, pendant, chain and {metal} finish identical to Picture 2, at realistic true-to-life scale with natural "
    "metallic reflections. Keep her face, identity, hairstyle, outfit, pose, lighting and background unchanged. "
    "Photorealistic jewellery catalogue photograph."),
"earrings": (
    "Put the exact earrings from Picture 2 on both earlobes of the woman in Picture 1. Keep the earring design and {metal} "
    "finish identical to Picture 2, realistic small size, natural metallic shine. Keep everything else unchanged. "
    "Photorealistic close-up jewellery photograph."),
"ring": (
    "Make the person in Picture 1 wear the exact ring from Picture 2 on the ring finger of the hand, clearly visible. "
    "Same design, stones and {metal} finish as Picture 2, realistic size. Keep everything else unchanged. "
    "Photorealistic jewellery catalogue photograph."),
"bangles": (
    "Make the woman in Picture 1 wear the exact bangles from Picture 2 on her wrist, clearly visible. Same design and "
    "{metal} finish as Picture 2, realistic size and shine. Keep everything else unchanged. Photorealistic catalogue photograph."),
}
DEFAULT_TYPE  = "necklace_set"
DEFAULT_METAL = "polished yellow gold with white gold accents"

## 6. Build the job list

**Option A: `jobs.csv`** (recommended once you have real shop products). Columns:
`model, jewellery, type, metal, prompt, seed`. Only `model` and `jewellery` (file names) are required; empty `prompt` means the template is used.
```
model,jewellery,type,metal,prompt,seed
woman_01.jpg,set_101.jpg,necklace_set,22k yellow gold,,
woman_02.jpg,ring_07.png,ring,rose gold with diamonds,,123
```
**Option B: no CSV.** Every jewellery image is paired with every model image (10 jewellery × 5 models = 50 images).
The type is guessed from the file name (`ring`, `earring`, `bangle`, `necklace`, otherwise `necklace_set`).

In [ ]:
import pandas as pd
IMG_EXT = (".jpg", ".jpeg", ".png", ".webp")

def find_dir(name):
    hits = [p for p in glob.glob(f"{INPUT_ROOT}/**/{name}", recursive=True) if os.path.isdir(p)]
    return hits[0] if hits else None

MODELS_DIR, JEWEL_DIR = find_dir("models"), find_dir("jewellery") or find_dir("jewelry")
assert MODELS_DIR and JEWEL_DIR, "Upload a dataset with 'models/' and 'jewellery/' folders (see top of notebook)"
list_imgs = lambda d: sorted(f for f in os.listdir(d) if f.lower().endswith(IMG_EXT))
print("models   :", MODELS_DIR, len(list_imgs(MODELS_DIR)))
print("jewellery:", JEWEL_DIR, len(list_imgs(JEWEL_DIR)))

def guess_type(fname):
    f = fname.lower()
    for key, t in [("earring", "earrings"), ("ring", "ring"), ("bangle", "bangles"), ("bracelet", "bangles"),
                   ("set", "necklace_set"), ("necklace", "necklace"), ("pendant", "necklace")]:
        if key in f:
            return t
    return DEFAULT_TYPE

csvs = glob.glob(f"{INPUT_ROOT}/**/jobs.csv", recursive=True)
if csvs:
    jobs = pd.read_csv(csvs[0]).fillna("")
    print("using", csvs[0])
else:
    jobs = pd.DataFrame([{"model": m, "jewellery": j} for j in list_imgs(JEWEL_DIR) for m in list_imgs(MODELS_DIR)])
for col in ["type", "metal", "prompt", "seed"]:
    if col not in jobs:
        jobs[col] = ""
jobs["type"]  = [t or guess_type(j) for t, j in zip(jobs["type"], jobs["jewellery"])]
jobs["metal"] = [m or DEFAULT_METAL for m in jobs["metal"]]
jobs["prompt"] = [p or PROMPT_TEMPLATES[t].format(metal=m) for p, t, m in zip(jobs["prompt"], jobs["type"], jobs["metal"])]
jobs["seed"]  = [int(s) if str(s).strip() else BASE_SEED for s in jobs["seed"]]
print(len(jobs), "jobs"); jobs.head()

## 7. Generation helpers

In [ ]:
import json, time
from PIL import Image, ImageOps

def load_rgb(path):
    return ImageOps.exif_transpose(Image.open(path)).convert("RGB")

def out_size(img, area=TARGET_PIXELS, mult=16):
    # keep the model photo's aspect ratio. Dimensions must be multiples of 16
    w, h = img.size
    s = math.sqrt(area / (w * h))
    return max(mult, round(w * s / mult) * mult), max(mult, round(h * s / mult) * mult)

def generate(model_img, jewel_img, prompt, seed, n=1):
    W, H = out_size(model_img)
    with torch.inference_mode():
        return pipe(
            image=[model_img, jewel_img],     # Picture 1 = model, Picture 2 = jewellery
            prompt=prompt, negative_prompt=" ",
            width=W, height=H,
            num_inference_steps=STEPS, true_cfg_scale=CFG, guidance_scale=1.0,
            num_images_per_prompt=n,
            generator=torch.Generator("cpu").manual_seed(seed),
        ).images

def side_by_side(*imgs, h=512):
    imgs = [i.resize((int(i.width * h / i.height), h)) for i in imgs]
    canvas = Image.new("RGB", (sum(i.width for i in imgs), h), "white")
    x = 0
    for i in imgs:
        canvas.paste(i, (x, 0)); x += i.width
    return canvas

## 8. Test ONE job first: check quality and timing before running hundreds

In [ ]:
r = jobs.iloc[0]
m, j = load_rgb(f"{MODELS_DIR}/{r.model}"), load_rgb(f"{JEWEL_DIR}/{r.jewellery}")
print(r.prompt, "\n")
t = time.time()
res = generate(m, j, r.prompt, int(r.seed))
print(f"⏱  {time.time()-t:.0f}s per image  |  peak VRAM {torch.cuda.max_memory_allocated()/1e9:.1f} GB")
res[0].save(f"{OUTPUT_DIR}/_test.png")
display(side_by_side(j, m, res[0]))

## 9. Batch run (can resume)
Images that already exist in `OUTPUT_DIR` are skipped, so if the session dies you can run this cell again and it continues where it stopped.

**For long runs:** click **Save Version → "Save & Run All (Commit)"**. Kaggle then runs the whole notebook in the background (up to 12 h)
and you can close the browser. Results appear under the version's **Output** tab.

In [ ]:
from tqdm.auto import tqdm
stem = lambda f: os.path.splitext(f)[0]
log_path = f"{OUTPUT_DIR}/metadata.jsonl"
done, failed, t0 = 0, [], time.time()

for idx, r in tqdm(jobs.iterrows(), total=len(jobs)):
    base = f"{stem(r.jewellery)}__{stem(r.model)}__s{r.seed}"
    if all(os.path.exists(f"{OUTPUT_DIR}/{base}_{k}.png") for k in range(IMAGES_PER_JOB)):
        continue
    try:
        m, j = load_rgb(f"{MODELS_DIR}/{r.model}"), load_rgb(f"{JEWEL_DIR}/{r.jewellery}")
        imgs = generate(m, j, r.prompt, int(r.seed), n=IMAGES_PER_JOB)
        for k, im in enumerate(imgs):
            im.save(f"{OUTPUT_DIR}/{base}_{k}.png")
        with open(log_path, "a") as f:
            f.write(json.dumps({"file": base, "model": r.model, "jewellery": r.jewellery, "type": r.type,
                                "prompt": r.prompt, "seed": int(r.seed), "steps": STEPS, "cfg": CFG}) + "\n")
        done += 1
    except torch.cuda.OutOfMemoryError:
        print("OOM on", base, "(lower TARGET_PIXELS)"); failed.append(base)
        gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        print("failed", base, e); failed.append(base)

print(f"✅ {done} new jobs in {(time.time()-t0)/60:.1f} min, {len(failed)} failed")

## 10. Preview + download

In [ ]:
outs = sorted(glob.glob(f"{OUTPUT_DIR}/*__*.png"))[:12]
if outs:
    thumbs = [Image.open(p) for p in outs]
    for i in range(0, len(thumbs), 4):
        display(side_by_side(*thumbs[i:i+4], h=384))

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/jewellery_outputs", "zip", OUTPUT_DIR)
print("Download /kaggle/working/jewellery_outputs.zip from the Output panel on the right →")

## Troubleshooting
| Problem | Fix |
|---|---|
| `CUDA out of memory` | Lower `TARGET_PIXELS` to `768*768`, keep `IMAGES_PER_JOB=1`, restart the kernel. |
| Black / NaN image | Set `FP16_ATTENTION_ON_OLD_GPUS=False` and lower `TARGET_PIXELS` instead. |
| Jewellery looks different from the product | Turn off Lightning (`USE_LIGHTNING=False`, 30 steps, CFG 4), and crop the product photo tightly on a plain background. |
| Pendant too big / plastic look | Add "delicate, realistic true-to-life scale, polished metal reflections" to the prompt. |
| Face changes | Add "keep her face and identity exactly unchanged". Use a sharp, front-facing model photo. |
| Lightning LoRA fails to load | Update diffusers (cell 1), or run without it. |
| Disk full | `BACKEND="nf4"` needs ~17 GB, `"gguf"` ~30 GB. Delete `/kaggle/working` leftovers. |